# 03_state_extraction_and_plots
Run NLP vs LLM state extraction, evaluate, and plot comparisons.

In [1]:
# If needed:
# !pip install -U pandas numpy scikit-learn matplotlib jsonschema openai
print('Notebook ready.')

Notebook ready.


In [2]:
from pathlib import Path
import json

data_dir = Path('../data/synthetic_data')
sample = sorted(data_dir.glob('transcript_*.json'))[0]
with open(sample, 'r', encoding='utf-8') as f:
    tr = json.load(f)
print('Sample intent:', tr['gt_primary_intent'])
print('Scenario:', tr['gt_scenario_family'])
print('Tool failure:', tr['gt_tool_failure'])
print('Turn count:', tr['gt_turn_count'])

Sample intent: CARD_REPLACEMENT
Scenario: clarify_then_resolve
Tool failure: False
Turn count: 7


In [9]:
# Run NLP state extraction (offline)
!python ../src/state_extraction_pipeline.py --input-dir ../data/synthetic_data --output-dir ../outputs/state_nlp --provider nlp --limit 5000

# Evaluate NLP extraction
!python ../src/eval_state_extraction.py --pred-dir ../outputs/state_nlp --output-dir ../results/state_nlp

Processed 5000 files to ../outputs/state_nlp
{
  "n_samples": 5000,
  "provider": null,
  "model": null,
  "primary_intent": {
    "accuracy": 0.3152,
    "macro_precision": 0.3520334584356905,
    "macro_recall": 0.22329215417537376,
    "macro_f1": 0.218108908576944
  },
  "secondary_intents": {
    "precision": 0.0,
    "recall": 0.0,
    "f1": 0.0,
    "exact_match": 0.5244
  },
  "multi_intent_accuracy": 0.5244,
  "tool_failure_accuracy": 1.0,
  "ambiguity": {
    "mae": 0.1548158,
    "rmse": 0.18651398178152756,
    "pearson": NaN
  },
  "sentiment_overall": {
    "mae": 0.36861298,
    "rmse": 0.41260602273112784,
    "pearson": NaN
  },
  "turn_count_mae": 0.0,
  "failure_count_mae": 0.0,
  "schema_valid_rate": 1.0,
  "average_jaccard_secondary": 0.5244
}
Saved summary + rows to /Users/manaswitamandal/Desktop/Mtech project/mtech-voicebot-context-project/results/state_nlp


In [15]:
# Optional: run LLM extraction if you have API access
!python ../src/state_extraction_pipeline.py --input-dir ../data/synthetic_data --output-dir ../outputs/state_ensemble_ollama --provider ollama --model qwen2.5:7b-instruct --limit 5000
!python ../src/eval_state_extraction.py --pred-dir ../outputs/state_ensemble_ollama --output-dir ../results/state_ensemble_ollama
# print('LLM mode is optional; NLP mode is already runnable offline.')

Extracting States: 100%|█| 5000/5000 [10:19:11<00:00,  7.43s/conversation, file
Processed 5000 files to ../outputs/state_ensemble_ollama
{
  "n_samples": 5000,
  "provider": null,
  "model": null,
  "primary_intent": {
    "accuracy": 0.778,
    "macro_precision": 0.5181357483690069,
    "macro_recall": 0.4886197732861973,
    "macro_f1": 0.4909982908397044
  },
  "secondary_intents": {
    "precision": 0.4460893854748603,
    "recall": 0.6088448341593595,
    "f1": 0.5149121392874415,
    "exact_match": 0.5848
  },
  "multi_intent_accuracy": 0.8222,
  "tool_failure_accuracy": 1.0,
  "ambiguity": {
    "mae": 0.16094160000000002,
    "rmse": 0.20926652957412947,
    "pearson": 0.11209081853179526
  },
  "sentiment_overall": {
    "mae": 0.40174722,
    "rmse": 0.4963533680695639,
    "pearson": 0.0022449210956669986
  },
  "turn_count_mae": 0.0,
  "failure_count_mae": 0.0,
  "schema_valid_rate": 1.0,
  "average_jaccard_secondary": 0.6252333333333334
}
Saved summary + rows to /Users/man

In [17]:
# Plot comparison for one or more methods
# Example:
!python ../src/plot_state_comparison.py \
  --summaries ../results/state_nlp/state_eval_summary.json ../results/state_ensemble_ollama/state_eval_summary.json \
  --rows ../results/state_nlp/state_eval_rows.csv ../results/state_ensemble_ollama/state_eval_rows.csv \
  --labels NLP LLM_NLP_ensemble \
  --output-dir ../results/state_plots
# print('Use plot_state_comparison.py after you have one or more evaluation summaries.')

Matplotlib is building the font cache; this may take a moment.
Saved plots and tables to /Users/manaswitamandal/Desktop/Mtech project/mtech-voicebot-context-project/results/state_plots
